In [1]:
class RuleBasedChatbot:
    def __init__(self):
        # 인간(기획자/개발자)이 정한 규칙 테이블 (IF-THEN 구조)
        self.rules = {
            "배송": "🚚 배송은 영업일 기준 2~3일이 소요됩니다. 운송장은 마이페이지에서 확인 가능합니다.",
            "환불": "💵 환불은 상품 수거 완료 후 3일 이내에 결제하신 수단으로 처리됩니다.",
            "운영시간": "⏰ 고객센터 운영 시간은 평일 오전 9시부터 오후 6시까지입니다. (주말/공휴일 제외)",
            "위치": "📍 저희 매장은 서울시 영등포구에 위치해 있습니다.",
            "인사": "안녕하세요! 무엇을 도와드릴까요? (배송, 환불, 운영시간, 위치 등)"
        }

    def respond(self, user_message: str) -> str:
        # 사용자가 입력한 메시지에서 공백을 제거하여 정제
        clean_message = user_message.replace(" ", "")

        # 1. 인사말 키워드 체크 규칙
        if "안녕" in clean_message or "하이" in clean_message:
            return self.rules["인사"]

        # 2. 본문 키워드 매칭 규칙 (루프를 돌며 규칙 매칭)
        for keyword, reply in self.rules.items():
            if keyword in clean_message:
                return reply

        # 3. 예외 처리 규칙 (일치하는 규칙이 없을 때)
        return "❓ 죄송합니다. 입력하신 내용을 이해하지 못했습니다. '배송', '환불', '운영시간' 등의 키워드로 문의해 주세요."

# 🚀 시스템 실행 테스트
if __name__ == "__main__":
    chatbot = RuleBasedChatbot()
    print("🤖 규칙 기반 챗봇이 시작되었습니다. (종료하려면 '종료' 입력)")
    print("-" * 50)
    
    while True:
        user_input = input("나: ")
        if user_input == "종료":
            print("🤖 챗봇을 종료합니다.")
            break
            
        bot_response = chatbot.respond(user_input)
        print(f"챗봇: {bot_response}")
        print("-" * 50)



🤖 규칙 기반 챗봇이 시작되었습니다. (종료하려면 '종료' 입력)
--------------------------------------------------
챗봇: 🚚 배송은 영업일 기준 2~3일이 소요됩니다. 운송장은 마이페이지에서 확인 가능합니다.
--------------------------------------------------
챗봇: ❓ 죄송합니다. 입력하신 내용을 이해하지 못했습니다. '배송', '환불', '운영시간' 등의 키워드로 문의해 주세요.
--------------------------------------------------
챗봇: ❓ 죄송합니다. 입력하신 내용을 이해하지 못했습니다. '배송', '환불', '운영시간' 등의 키워드로 문의해 주세요.
--------------------------------------------------
챗봇: ❓ 죄송합니다. 입력하신 내용을 이해하지 못했습니다. '배송', '환불', '운영시간' 등의 키워드로 문의해 주세요.
--------------------------------------------------
챗봇: ❓ 죄송합니다. 입력하신 내용을 이해하지 못했습니다. '배송', '환불', '운영시간' 등의 키워드로 문의해 주세요.
--------------------------------------------------
챗봇: ❓ 죄송합니다. 입력하신 내용을 이해하지 못했습니다. '배송', '환불', '운영시간' 등의 키워드로 문의해 주세요.
--------------------------------------------------
챗봇: ❓ 죄송합니다. 입력하신 내용을 이해하지 못했습니다. '배송', '환불', '운영시간' 등의 키워드로 문의해 주세요.
--------------------------------------------------
챗봇: ❓ 죄송합니다. 입력하신 내용을 이해하지 못했습니다. '배송', '환불', '운영시간' 등의 키워드로 문의해 주세요.
------------

In [4]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

class MLChatbot:
    def __init__(self):
        # 1. delivery 관련 질문 (총 12개)
        delivery_questions = [
            "배송 언제 되나요", "택배 언제 와요", "배송일 조회", "언제 출고되나요",
            "배송", "택배", "배쏭", "택배 언제", "언제와요", "언제와", "조회해줘",
            "물건 언제 도착하나요", "배송 상태", "출고 완료", "언제 보내주나요"
        ]
        
        # 2. refund 관련 질문 (총 10개)
        refund_questions = [
            "환불하고 싶어요", "반품 처리 해주세요", "돈 돌려받나요", "취소 가능한가요",
            "환불", "반품", "취소", "환불해줘", "반품할래요", "결제 취소"
        ]
        
        # 3. hours 관련 질문 (총 11개)
        hours_questions = [
            "몇 시까지 하나요", "운영시간 알려줘", "영업시간 궁금해요", "언제 문 여나요",
            "운영시간", "영업시간", "몇시까지", "오픈 시간", "마감 시간", "언제 열어", "언제 닫아"
        ]
        
        # 4. location 관련 질문 (총 10개)
        location_questions = [
            "어디에 있나요", "위치가 어디죠", "매장 주소 알려주세요", "찾아가는 길",
            "위치", "주소", "매장", "어디에요", "오프라인 매장", "위치 안내"
        ]

        # 모든 질문 리스트 합치기 (총 43개)
        self.train_questions = delivery_questions + refund_questions + hours_questions + location_questions
        
        # 각 질문 개수에 정확하게 일치하도록 라벨 생성 (총 43개)
        self.train_labels = (
            ["delivery"] * len(delivery_questions) + 
            ["refund"] * len(refund_questions) + 
            ["hours"] * len(hours_questions) + 
            ["location"] * len(location_questions)
        )

        self.responses = {
            "delivery": "🚚 배송은 영업일 기준 2~3일이 소요됩니다.",
            "refund": "💵 환불은 상품 수거 완료 후 3일 이내에 처리됩니다.",
            "hours": "⏰ 고객센터 운영 시간은 평일 오전 9시~오후 6시입니다.",
            "location": "📍 저희 매장은 서울시 영등포구에 위치해 있습니다."
        }

        # 글자 단위(char_wb)로 2~3글자씩 쪼개서 오타 패턴을 감지하도록 설정
        self.vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 3))
        self.model = MultinomialNB()
        self._train_model()

    def _train_model(self):
        X_train = self.vectorizer.fit_transform(self.train_questions)
        self.model.fit(X_train, self.train_labels)

    def respond(self, user_message: str) -> str:
        user_vector = self.vectorizer.transform([user_message])
        probabilities = self.model.predict_proba(user_vector)
        max_prob_idx = np.argmax(probabilities)
        max_probability = probabilities[0][max_prob_idx] # 2차원 배열이므로 첫 번째 행 지정
        predicted_intent = self.model.classes_[max_prob_idx]

        # 디버깅 출력 (예측 정보 확인용)
        print(f"[의도 예측: {predicted_intent} / 확신도: {max_probability:.2f}]")

        if max_probability < 0.35: 
            return "❓ 죄송합니다. 질문의 의도를 정확히 이해하지 못했습니다."

        return self.responses[predicted_intent]

if __name__ == "__main__":
    chatbot = MLChatbot()
    print("🤖 오류가 수정되고 오타 보정이 강화된 챗봇이 시작되었습니다.")
    print("-" * 60)
    while True:
        user_input = input("나: ")
        if user_input == "종료": break
        print(f"챗봇: {chatbot.respond(user_input)}\n" + "-"*50)



🤖 오류가 수정되고 오타 보정이 강화된 챗봇이 시작되었습니다.
------------------------------------------------------------
[의도 예측: delivery / 확신도: 0.55]
챗봇: 🚚 배송은 영업일 기준 2~3일이 소요됩니다.
--------------------------------------------------
[의도 예측: hours / 확신도: 0.30]
챗봇: ❓ 죄송합니다. 질문의 의도를 정확히 이해하지 못했습니다.
--------------------------------------------------
